In [1]:
# import sys
# !{sys.executable} -m pip install --upgrade --force-reinstall datasets pyarrow huggingface-hub
# !pip install -q transformers accelerate peft trl bitsandbytes sentencepiece transformers

In [2]:
!pip install -U transformers==4.56.1 trl==0.21.0 peft==0.17.1 accelerate==1.10.1

In [3]:
import trl
import peft
import torch
import numpy as np
import transformers
import bitsandbytes

from inspect import signature

from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

from trl import SFTTrainer

from peft import LoraConfig, get_peft_model

print("Imports Successful!")

Imports Successful!


In [4]:
print(f"TRL Version: {trl.__version__}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA:", torch.version.cuda)
print("BF16 Supported:", torch.cuda.is_bf16_supported())
print(signature(SFTTrainer.__init__))

if torch.cuda.is_available():
  print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
  print("CUDA not available, running on CPU")

TRL Version: 0.21.0
PyTorch Version: 2.10.0+cu128
CUDA Available: True
Transformers: 4.56.1
BitsAndBytes: 0.49.2
CUDA: 12.8
BF16 Supported: True
(self, model: Union[str, torch.nn.modules.module.Module, transformers.modeling_utils.PreTrainedModel], args: Union[trl.trainer.sft_config.SFTConfig, transformers.training_args.TrainingArguments, NoneType] = None, data_collator: Optional[transformers.data.data_collator.DataCollator] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, dict[str, datasets.arrow_dataset.Dataset], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.FeatureExtractionMixin, transformers.processing_utils.ProcessorMixin, NoneType] = None, compute_loss_func: Optional[Callable] = None, compute_metrics: Option

In [5]:
dataset = load_dataset("openai/gsm8k", "main")
print("Dataset Imported Successfully!")

Dataset Imported Successfully!


In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [7]:
# Check the dataset sample
sample = dataset["train"][0]

print(sample)

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [8]:
# Pretty Printing Dataset Sample
print("QUESTION:")
print(sample["question"])

print("\n" + "="*80 + "\n")

print("ANSWER:")
print(sample["answer"])

QUESTION:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?


ANSWER:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [9]:
# Dataset Exploration
print(f"Length of Train Dataset: {len(dataset["train"])}")
print("=" * 300)
print(f"Length of Test Dataset: {len(dataset["test"])}")
print("=" * 300)
print(f"Columns of Train Dataset: {dataset["train"].column_names}")
print("=" * 300)
print(f"Columns of Test Dataset: {dataset["test"].column_names}")
print("=" * 300)
# Exact structure of data fed to the model
print(dataset["train"][10])

Length of Train Dataset: 7473
Length of Test Dataset: 1319
Columns of Train Dataset: ['question', 'answer']
Columns of Test Dataset: ['question', 'answer']
{'question': 'A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hunger. Over three hundred years, it has consumed 847 people. Ships have been built larger over time, so each new ship has twice as many people as the last ship. How many people were on the ship the monster ate in the first hundred years?', 'answer': 'Let S be the number of people on the first hundred years’ ship.\nThe second hundred years’ ship had twice as many as the first, so it had 2S people.\nThe third hundred years’ ship had twice as many as the second, so it had 2 * 2S = <<2*2=4>>4S people.\nAll the ships had S + 2S + 4S = 7S = 847 people.\nThus, the ship that the monster ate in the first hundred years had S = 847 / 7 = <<847/7=121>>121 people on it.\n#### 121'}


In [10]:
# Defining the model we will use for fine tuning
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer Loaded Successfully!")

Tokenizer Loaded Successfully!


In [12]:
# Checking the tokenizer information
print(tokenizer)

Qwen2TokenizerFast(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False,

In [13]:
# Testing the tokenizer
text = "John has 25 apples."

tokens = tokenizer(text)

print(tokens)

{'input_ids': [13079, 702, 220, 17, 20, 40676, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [14]:
# Decoding the token back to original text
decoded = tokenizer.decode(tokens["input_ids"])

print(decoded)

John has 25 apples.


In [15]:
# Counting the tokens
print(len(tokens['input_ids']))

7


In [16]:
# Tokenizing a GSM8K Question
question = dataset['train'][0]['question']
print(question)

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?


In [17]:
# Encoding the Question
encode = tokenizer(question)
print(encode['input_ids'][:50])
print("=" * 300)
print(f"Number of Tokens: {len(encode['input_ids'])}")

[45, 4212, 685, 6088, 26111, 311, 220, 19, 23, 315, 1059, 4780, 304, 5813, 11, 323, 1221, 1340, 6088, 4279, 438, 1657, 26111, 304, 3217, 13, 2585, 1657, 26111, 1521, 41601, 685, 4559, 30055, 304, 5813, 323, 3217, 30]
Number of Tokens: 39


#**Why We are Creating a Formatting Function when we can pass the model with Question and Answer Only?**
Model we are using is trained on conversations not plain text. Instead of Question -> Answer, we want System [Instruction] -> User [Question] -> Assistant [Answer]. If we train on Question and Answer only then Model will forget that its supposed to behave like an assistant. Instruction Tuning preserves its conversational behavior

In [18]:
# Verifying that tokenizer supports chat templates
print(tokenizer.chat_template is not None)

True


In [19]:
# Creating a formatting function
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful math tutor. Solve the user's math problem step by step."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [20]:
# Applying the Formatting Function to the Dataset
formatted_dataset = dataset.map(format_example)

In [21]:
# Inpect on one formatted sample
print(formatted_dataset['train'][0]['text'])

<|im_start|>system
You are a helpful math tutor. Solve the user's math problem step by step.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



In [22]:
# Verifying the formatting
print(formatted_dataset["train"][1]["text"])
print("=" * 300)
print(formatted_dataset["train"][25]["text"])
print("=" * 300)
print(formatted_dataset["train"][100]["text"])

<|im_start|>system
You are a helpful math tutor. Solve the user's math problem step by step.<|im_end|>
<|im_start|>user
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?<|im_end|>
<|im_start|>assistant
Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10<|im_end|>

<|im_start|>system
You are a helpful math tutor. Solve the user's math problem step by step.<|im_end|>
<|im_start|>user
Ralph is going to practice playing tennis with a tennis ball machine that shoots out tennis balls for Ralph to hit. He loads up the machine with 175 tennis balls to start with. Out of the first 100 balls, he manages to hit 2/5 of them. Of the next 75 tennis balls, he manages to hit 1/3 of them. Out of all the tennis balls, how many did Ralph not hit?<|im_end|>
<|im_start|>assistant
Out of the first 100 balls, Ralph was able to hit 2/5 of them and not able to hit 3/5 of them, 3/5 x 

# **Why we decide maximum length?**
Every model has a maximum context length. For GSM8K, 512 tokens is sufficient for most examples. Later, if we discover many samples are longer, we'll increase it to 768 or 1024.



---

`truncation=True`
If an example exceeds 512 tokens:

700 tokens

↓

Keep first 512


---

`padding="max_length"`

Suppose one sample has: 250 tokens

Another has: 480 tokens

The model requires tensors of equal length.

So the first becomes: 250 real tokens, 262 padding tokens

Total = 512.

Those padding tokens are ignored because of the attention mask.



---

`max_length=512`

Every example becomes exactly

512 tokens

This simplifies batching during training.

In [23]:
# Tokenizing the Entire Dataset
MAX_LENGTH = 512

# Tokenization Function
def tokenize_function(example):
  return tokenizer(
      example['text'],
      truncation=True,
      max_length=MAX_LENGTH
  )

In [24]:
# Tokenize Everything
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=False
)

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [25]:
# Inspecting the results
tokenized_dataset["train"][0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72',
 'text': "<|im_start|>system\nYou are a helpful math tutor. Solve the user's math problem step by step.<|im_end|>\n<|im_start|>user\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\nNatalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72<|im_end|>\n",
 'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  10950,
  6888,
  25302,
  13,
  63284,
  279,
  1196,
  594,
  6888,
  3491,
  3019,
  553,
  3019,
  13,
  151645,
  198,
  151644,
  872,
  198,
  45,
  42

In [26]:
# Check token length
sample = tokenized_dataset["train"][0]
print("Input IDs:", len(sample["input_ids"]))
print("Attention Mask:", len(sample["attention_mask"]))

Input IDs: 130
Attention Mask: 130


In [27]:
# Decoding Back
decoded = tokenizer.decode(
    tokenized_dataset["train"][0]["input_ids"],
    skip_special_tokens=False
)

print(decoded)

<|im_start|>system
You are a helpful math tutor. Solve the user's math problem step by step.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



In [28]:
# Checking for Truncation - we do not want to accidentally cut off many answers
lengths = []

for example in formatted_dataset["train"]:
  tokens = tokenizer(example["text"], truncation=False)
  lengths.append(len(tokens["input_ids"]))

# Now inspecting them
print(f"Shortest: {min(lengths)}")
print("=" * 300)
print(f"Longest: {max(lengths)}")
print("=" * 300)
print(f"Average: {sum(lengths) / len(lengths)}")

# Now checking the percentiles
print("90th percentile:", np.percentile(lengths, 90))
print("=" * 300)
print("95th percentile:", np.percentile(lengths, 95))
print("=" * 300)
print("99th percentile:", np.percentile(lengths, 99))

Shortest: 96
Longest: 568
Average: 213.44051920246218
90th percentile: 307.0
95th percentile: 344.0
99th percentile: 422.0


#**LoRA (Low Rank Adaption) and Quantization**
##**The Problem**
Our model has about 1.5 billion parameters. Training all of them requires tens of gigabytes of GPU memory. A free Google Colab T4 has only 16 GB VRAM. Training the whole model is impossible.

##**Solution 01 - LoRA**
LoRA (Low-Rank Adaptation) freezes almost all of the model. Think of the model like a huge textbook. Instead of rewriting every page, LoRA adds a few sticky notes with new information.

Original Model (Frozen) ---> Original Weights |OR|
        -
        -
        -
        >
    Small LoRA Layers
        -
        -
        >
    Train Only These

Instead of training: **1,500,000,000** parameters

We train roughly: **5-15 million** parameters

**Training becomes:**

1.   Much Faster
2.   Uses Much Less GPU Memory
3.   Nearly the same quality


This is how many real-world LLM fine-tuning pipelines work.

##**Solution 02 - Quantization**
Normally every parameter is stored as a 16-bit number. 16 bits x 1.5 billion parameters Huge memory usage. Instead we compress them.

16-bit

  ↓

4-bit

The model becomes roughly 4x smaller. Quality drops very little. Memory usage drops dramatically.

In [29]:
# Configuring 4-Bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# **What do these mean?**
`load_in_4bit=True`

Load the model in compressed 4-bit format.

`nf4`

Normal Float 4.

This is the best quantization format for training.

`float16`

During calculations, temporarily use float16.

Faster on NVIDIA GPUs.

`double_quant=True`

Compress the compressed weights again.

Less memory.

Same accuracy.

In [30]:
# Loading the Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [31]:
# Checking if the model is loaded
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps

In [32]:
# Configuring LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

# **What do these mean?**



---


`r`

Rank.

Think of it as LoRA capacity.

Higher = learns more.

Higher = more GPU memory.

16 is a good starting point.



---



`alpha`

Scaling factor.

Controls how strongly LoRA affects the model.



---



`dropout`

Helps prevent overfitting.

5% is common.



---



`target_modules`

The transformer contains many layers.

We only inject LoRA into:

Query projection
Key projection
Value projection
Output projection



---



These are the attention layers.

They are the most useful places for fine-tuning.

In [33]:
# Attaching LoRA
model = get_peft_model(model, lora_config)

Now the Model Becomes : Original Model + LoRA Configs

In [34]:
# Verifying Trainable Parameters
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [35]:
# Preparing the Model for Training
model.config.use_cache = False
model.enable_input_require_grads()

## **Why?**

During inference, transformers cache previous tokens to generate text faster.

During training, caching interferes with gradient computation.

So we disable it.

In [36]:
# Configuring Training
training_args = TrainingArguments(
    output_dir="./math_tutor_qwen",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

# **Let's understand every parameter**
`output_dir`

Where checkpoints are saved.

math_tutor_qwen/

checkpoint-100/

checkpoint-200/



---



`num_train_epochs`

How many complete passes through GSM8K.

For our first experiment:

2

Later we'll experiment with

3
4
5



---



`batch_size=2`

Each GPU step sees only

2 samples

Why?

Memory.

The T4 cannot fit large batches.



---



`gradient_accumulation_steps=4`

This is extremely important.

Actual effective batch size becomes

2 x 4 = 8

The GPU processes

2

2

2

2

Then updates once.

This simulates batch size 8 without needing extra VRAM.



---



`learning_rate`
2e-4

A standard LoRA learning rate.



---



`fp16=True`

Mixed precision.

Almost

1. 2x faster
2. less memory

In [37]:
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)

transformers: 4.56.1
trl: 0.21.0
peft: 0.17.1
torch: 2.10.0+cu128


In [38]:
# Creating the Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

# **Why formatted_dataset and not tokenized_dataset?**

The latest SFTTrainer tokenizes internally. It expects the text field and the tokenizer (processing_class).

In [ ]:
# Starting the Training
model.config.use_cache = False
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
